# Module 1 · Lesson 05: Token Explorer

**Tokens are the currency of LLMs.** Every API call is billed per token, and every model
has a maximum token limit (context window). Understanding tokenization is essential.

## What you will learn
1. What tokens are and how text is tokenized
2. How to **count** tokens before making an API call
3. **Cost estimation** for any prompt
4. Context window limits
5. How tokenization differs across languages and code

---
### Key Library: `tiktoken`
OpenAI's open-source tokenizer. It lets you count tokens *locally* — no API call needed!

In [1]:
import tiktoken 
from IPython.display import Markdown, display

enc = tiktoken.encoding_for_model('gpt-4o-mini')
print(f"Encoder: {enc.name}")
print(f"    Vocabulary size: {enc.n_vocab:,} tokens")

Encoder: o200k_base
    Vocabulary size: 200,019 tokens


---
## 1. Token Visualization

Let's see how text is broken into tokens. Each token is a piece of text (could be a word, part of a word, or punctuation).

In [4]:
text = "Hello World! Tokenization is fascinating"

tokens = enc.encode(text)
print(f"Text: {text}")
print(f"Token count: {len(tokens)}")
print(f"Tokens IDs: {tokens}")

print("Token-by-token breakdown:")
print(f"{'#':<4} {'ID':<10} {'Token':<20} {'Bytes':<6}")
print("-" * 40)
for i, tok_id in enumerate(tokens):
    decoded = enc.decode([tok_id])
    print(f"{i+1:<4} {tok_id:<10} {repr(decoded):<20} {len(decoded):<6}")

Text: Hello World! Tokenization is fascinating
Token count: 7
Tokens IDs: [13225, 5922, 0, 17951, 2860, 382, 33733]
Token-by-token breakdown:
#    ID         Token                Bytes 
----------------------------------------
1    13225      'Hello'              5     
2    5922       ' World'             6     
3    0          '!'                  1     
4    17951      ' Token'             6     
5    2860       'ization'            7     
6    382        ' is'                3     
7    33733      ' fascinating'       12    


> 💡 **Key Insight:** Common words like "Hello" are usually 1 token.
> Rare or long words may be split into multiple tokens.
> Spaces are often included *with* the following word.

---
## 2. Cost Calculator

Knowing the token count lets us estimate costs *before* making the API call:

In [7]:
# ── Cost estimation ───────────────────────────────────
PRICING = {
    "gpt-4o-mini":  {"input": 0.15,  "output": 0.60},    # per 1M tokens
    "gpt-4o":       {"input": 2.50,  "output": 10.00},
    "claude-3-5-haiku": {"input": 1.00, "output": 5.00},
}

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def estimate_cost(input_text: str, output_ratio: float = 1.5) -> None:
    """Estimate cost for a prompt, assuming output is `output_ratio` × input length."""
    in_tokens = count_tokens(input_text)
    out_tokens = int(in_tokens * output_ratio)

    print(f"📝 Input:  {in_tokens} tokens")
    print(f"📤 Output: ~{out_tokens} tokens (estimated)")
    print()
    print(f"{'Model':<22} {'1 Call':>10} {'1K Calls':>12} {'1M Calls':>12}")
    print("-" * 58)

    for model, prices in PRICING.items():
        cost = (in_tokens / 1e6) * prices["input"] + (out_tokens / 1e6) * prices["output"]
        print(f"{model:<22} {cost:>9.6f}$ {cost*1000:>11.4f}$ {cost*1e6:>11.2f}$")

# Example
estimate_cost("Explain quantum computing to a high school student in detail.")

📝 Input:  11 tokens
📤 Output: ~16 tokens (estimated)

Model                      1 Call     1K Calls     1M Calls
----------------------------------------------------------
gpt-4o-mini             0.000011$      0.0112$       11.25$
gpt-4o                  0.000187$      0.1875$      187.50$
claude-3-5-haiku        0.000091$      0.0910$       91.00$


---
## 3. Context Windows

Every model has a **maximum context window** — the total tokens (input + output) it can handle.

| Model | Context Window | Approx. Pages |
|-------|---------------|----------------|
| gpt-4o-mini | 128K tokens | ~200 pages |
| gpt-4o | 128K tokens | ~200 pages |
| claude-3.5-sonnet | 200K tokens | ~300 pages |
| claude-3-opus | 200K tokens | ~300 pages |

In [8]:
# ── Context window demo ───────────────────────────────
CONTEXT_WINDOWS = {
    "gpt-4o-mini": 128_000,
    "gpt-4o": 128_000,
    "claude-3.5-sonnet": 200_000,
}

# How much can we fit?
sample = "The quick brown fox jumps over the lazy dog. "  # ~10 tokens
sample_tokens = count_tokens(sample)

print(f"Sample sentence: {sample_tokens} tokens\n")
for model, window in CONTEXT_WINDOWS.items():
    sentences = window // sample_tokens
    pages = window // 500  # ~500 tokens per page
    print(f"{model:<22} {window:>8,} tokens  ≈ {pages:>4} pages  ≈ {sentences:>6,} sentences")

Sample sentence: 11 tokens

gpt-4o-mini             128,000 tokens  ≈  256 pages  ≈ 11,636 sentences
gpt-4o                  128,000 tokens  ≈  256 pages  ≈ 11,636 sentences
claude-3.5-sonnet       200,000 tokens  ≈  400 pages  ≈ 18,181 sentences


---
## 4. Multilingual Tokenization

English text uses fewer tokens per word than other languages.
This directly impacts cost!

In [9]:
# ── Multilingual comparison ───────────────────────────
texts = {
    "English":    "Artificial intelligence is transforming the world.",
    "Greek":      "Η τεχνητή νοημοσύνη μεταμορφώνει τον κόσμο.",
    "Chinese":    "人工智能正在改变世界。",
    "Japanese":   "人工知能は世界を変えています。",
    "Arabic":     "الذكاء الاصطناعي يغير العالم.",
}

print(f"{'Language':<12} {'Chars':<8} {'Tokens':<8} {'Tokens/Char':<12} {'Cost Multiplier'}")
print("-" * 60)

en_tokens = count_tokens(texts["English"])
for lang, text in texts.items():
    n_chars = len(text)
    n_tokens = count_tokens(text)
    ratio = n_tokens / n_chars
    multiplier = n_tokens / en_tokens
    print(f"{lang:<12} {n_chars:<8} {n_tokens:<8} {ratio:<12.3f} {multiplier:.2f}×")

Language     Chars    Tokens   Tokens/Char  Cost Multiplier
------------------------------------------------------------
English      50       7        0.140        1.00×
Greek        43       17       0.395        2.43×
Chinese      11       6        0.545        0.86×
Japanese     15       10       0.667        1.43×
Arabic       29       11       0.379        1.57×


> ⚠️ **Non-Latin languages can cost 2–3× more** for the same semantic content!

---
## 5. Code vs Prose

In [10]:
# ── Code vs prose tokenization ────────────────────────
prose = """Python is a versatile programming language known for its readability.
It supports multiple paradigms including procedural, object-oriented, and functional programming."""

code = """def fibonacci(n: int) -> list[int]:
    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[i-1] + fib[i-2])
    return fib[:n]"""

for label, text in [("Prose", prose), ("Code", code)]:
    n_chars = len(text)
    n_tokens = count_tokens(text)
    print(f"{label}: {n_chars} chars → {n_tokens} tokens (ratio: {n_tokens/n_chars:.3f} tok/char)")

Prose: 167 chars → 26 tokens (ratio: 0.156 tok/char)
Code: 137 chars → 50 tokens (ratio: 0.365 tok/char)


---
## 6. Exercise — Token Budget Planner 🏋️

Calculate the monthly cost of your app:

In [ ]:
# ── YOUR TURN: Estimate your app's monthly cost ──────
# Configure your scenario:
calls_per_day = 1000
avg_input_tokens = 500
avg_output_tokens = 300
model = "gpt-4o"  # Try changing to "gpt-4o-mini"!

# Calculate
prices = PRICING[model]
cost_per_call = (avg_input_tokens / 1e6) * prices["input"] + (avg_output_tokens / 1e6) * prices["output"]
daily_cost = cost_per_call * calls_per_day
monthly_cost = daily_cost * 30

print(f"📊 Budget Estimate for {model}")
print(f"   Calls/day:    {calls_per_day:,}")
print(f"   Cost/call:    ${cost_per_call:.6f}")
print(f"   Daily cost:   ${daily_cost:.2f}")
print(f"   Monthly cost: ${monthly_cost:.2f}")
print(f"   Yearly cost:  ${monthly_cost * 12:.2f}")

📊 Budget Estimate for gpt-4o
   Calls/day:    1,000
   Cost/call:    $0.004250
   Daily cost:   $4.25
   Monthly cost: $127.50
   Yearly cost:  $1530.00


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **Token** | A chunk of text (word, subword, or punctuation) |
| **tiktoken** | Count tokens locally without API calls |
| **Cost formula** | `(input_tokens × input_price + output_tokens × output_price) / 1M` |
| **Context window** | Max tokens per request (input + output combined) |
| **Multilingual** | Non-Latin text costs more per semantic unit |
| **Budget planning** | Always estimate costs before building at scale |

---
**Next:** `06_streaming_responses.ipynb` — Real-time token delivery for responsive UIs